# 💰 The Gold Standard: Representing Money in Code

Welcome, **Initiates**, to one of the most practically important lessons in the Academy. Today we tackle a question that every wizard who has ever run a shop, managed a dragon's treasury, or built a trading system has had to answer:

> **How do you store money in code?**

It sounds trivial. Money is just a number, right? Pick `float` and move on?

This is the thinking of an apprentice—and it is the thinking that has caused real banks to lose real money, and real software systems to fail in ways that are very difficult to debug.

By the end of this lesson, you will understand *why* the obvious approach fails, and *three* progressively safer approaches that professional wizards use instead.

---

## ⚠️ 1. The Fool's Gold Approach: Using `float` Directly

The most natural instinct is to represent a price like `£12.99` as the float `12.99`. For simple display, this works fine. But the moment you start *calculating* with floats, the **Precision Curse** from our last lesson returns—and with money, its consequences are severe.

In [ ]:
# A merchant's simple shop
wand_price    = 9.99
potion_price  = 4.99
scroll_price  = 1.99

total = wand_price + potion_price + scroll_price

print(f"Wand:    £{wand_price:.2f}")
print(f"Potion:  £{potion_price:.2f}")
print(f"Scroll:  £{scroll_price:.2f}")
print(f"Total:   £{total:.2f}")  # Looks correct...
print(f"Exact:   £{total}")       # The ugly truth

The display with `:.2f` *looks* right, but the number stored in memory is `16.969999999999998`. That hidden error will compound with every further calculation.

### 🔥 Where This Becomes Dangerous

In [ ]:
# A tax calculation on 1000 transactions
item_price = 0.10   # 10 pence
tax_rate   = 0.07   # 7% guild tax

# Manually summing tax across 1000 items
total_tax = 0.0
for _ in range(1000):
    total_tax += item_price * tax_rate

expected = 1000 * item_price * tax_rate

print(f"Expected: £{expected:.10f}")
print(f"Actual:   £{total_tax:.10f}")
print(f"Match:    {total_tax == expected}")
print(f"Error:    £{abs(total_tax - expected):.2e}")

A tiny error per transaction, multiplied across thousands of trades, becomes a discrepancy that fails audits, breaks reconciliation, and—in regulated industries—can result in legal consequences.

> [!WARNING]
> **Never use raw `float` arithmetic for financial calculations** that will be stored, compared, or accumulated. Displaying a price as `f"{price:.2f}"` is fine; performing a chain of arithmetic on floats and trusting the result is not.

---

## 🥈 2. The Better Approach: `round()` as a Patch

A common intermediate step is to `round()` the result of every calculation back to 2 decimal places. This suppresses most visible errors and is acceptable for simple applications.

In [ ]:
wand_price   = 9.99
potion_price = 4.99
scroll_price = 1.99

# Round every intermediate result
subtotal = round(wand_price + potion_price + scroll_price, 2)
tax      = round(subtotal * 0.20, 2)   # 20% guild tax
total    = round(subtotal + tax, 2)

print(f"Subtotal: £{subtotal:.2f}")
print(f"Tax:      £{tax:.2f}")
print(f"Total:    £{total:.2f}")

This is better—but it is still a patch over a leaky cauldron. The rounding errors still exist underneath; we are just containing them. And because of **Banker's Rounding** (ties go to even), the rounded result can still surprise you in edge cases.

For personal projects and simple scripts, this is often good enough. For anything handling real transactions, you need one of the two approaches below.

---

## 🥇 3. The Wizard's Approach: Integer Pennies

Here is the elegant solution used by professional financial systems worldwide: **store all money as whole integers in the smallest currency unit**.

Instead of storing `£12.99` as the float `12.99`, store it as the integer `1299` — meaning *1299 pennies*. Since integers are exact, all arithmetic on integers is also exact. The Precision Curse cannot touch integers.

```
£12.99  →  store as  1299  (pennies)
£ 0.01  →  store as     1  (penny)
£99.00  →  store as  9900  (pennies)
```

To display the amount, we reconstruct the pounds and pence using the operators you already know from the integers lesson:

```python
pounds = amount // 100   # whole pounds
pence  = amount  % 100   # remaining pence
```

In [ ]:
def format_money(pennies):
    """Display an integer number of pennies as £X.XX"""
    pounds = pennies // 100
    pence  = pennies  % 100
    return f"£{pounds}.{pence:02d}"   # :02d ensures '05' not '5'


# Store every price as an integer in pennies
wand_price    = 999    # £9.99
potion_price  = 499    # £4.99
scroll_price  = 199    # £1.99

# All arithmetic is exact — integers have no precision curse!
subtotal = wand_price + potion_price + scroll_price
tax      = subtotal * 20 // 100   # 20% tax, floor division keeps it integer
total    = subtotal + tax

print(f"Wand:     {format_money(wand_price)}")
print(f"Potion:   {format_money(potion_price)}")
print(f"Scroll:   {format_money(scroll_price)}")
print(f"Subtotal: {format_money(subtotal)}")
print(f"Tax:      {format_money(tax)}")
print(f"Total:    {format_money(total)}")

### 🔍 Why `subtotal * 20 // 100` Instead of `subtotal * 0.20`?

If we wrote `subtotal * 0.20`, we would immediately introduce a float—and the curse returns. Instead we multiply by the integer `20` and then floor-divide by the integer `100`. The entire calculation stays in the land of integers, and is perfectly exact.

In [ ]:
# Demonstration: the same tax calculation, 1000 times over
item_pennies = 10    # 10p

total_tax_pennies = 0
for _ in range(1000):
    total_tax_pennies += item_pennies * 7 // 100   # 7% guild tax

expected_pennies = 1000 * item_pennies * 7 // 100

print(f"Expected: {format_money(expected_pennies)}")
print(f"Actual:   {format_money(total_tax_pennies)}")
print(f"Match:    {total_tax_pennies == expected_pennies}")
# True — integers are always exact

> [!NOTE]
> **The rounding policy for fractional pennies** (e.g. 7% of 10p = 0.7p) is a *business decision*, not a technical one. Floor division (`//`) always rounds down (in the customer's favour). You may need to round differently depending on the rules of your Guild. The important thing is that the policy is explicit and consistent.

---

## 🏆 4. The Grand Arsenal Approach: `decimal.Decimal`

For the most demanding financial work—accounting systems, regulated financial software, multi-currency conversions—Python provides the `decimal` module. `Decimal` numbers work like floats in terms of convenience, but use **base-10 arithmetic** internally, eliminating binary rounding errors entirely.

You can even set the **precision** (number of significant digits) and **rounding rule** globally for your entire programme.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, getcontext

# Set global precision and rounding rule
getcontext().prec    = 10
getcontext().rounding = ROUND_HALF_UP  # Classic "always round 0.5 up"

# IMPORTANT: always create Decimals from STRINGS, not floats!
wand_price   = Decimal("9.99")
potion_price = Decimal("4.99")
scroll_price = Decimal("1.99")
tax_rate     = Decimal("0.20")

subtotal = wand_price + potion_price + scroll_price
tax      = (subtotal * tax_rate).quantize(Decimal("0.01"))  # Round to 2dp
total    = subtotal + tax

print(f"Subtotal: £{subtotal}")
print(f"Tax:      £{tax}")
print(f"Total:    £{total}")
print(f"Exact?    {subtotal == Decimal('16.97')}")

> [!WARNING]
> **Never create a `Decimal` from a `float`!** `Decimal(9.99)` captures the float's binary error and is just as broken as using a float directly. Always pass a **string**: `Decimal("9.99")`.

### `.quantize()` — The Precision Ritual

`.quantize(Decimal("0.01"))` is the `Decimal` equivalent of rounding to 2 decimal places. It lets you control both the precision *and* the rounding rule at the point where rounding matters.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_DOWN, ROUND_UP

amount = Decimal("12.345")

# Different rounding modes for different business rules
print(amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))  # 12.35
print(amount.quantize(Decimal("0.01"), rounding=ROUND_DOWN))     # 12.34 — always favour the merchant
print(amount.quantize(Decimal("0.01"), rounding=ROUND_UP))       # 12.35 — always favour the guild

---

## 🗺️ 5. Choosing Your Approach: The Decision Map

| Situation | Recommended Approach |
| :--- | :--- |
| Displaying a price read from a database | `float` + `:.2f` formatting is fine |
| A simple script summing a short list of prices | `round()` at each step is acceptable |
| A till system, invoice generator, or game economy | **Integer pennies** — simple, fast, exact |
| Accounting, tax, regulated finance, multi-currency | **`decimal.Decimal`** — maximum correctness |

---

## 🧩 6. A Complete Integer-Penny Receipt System

Let us see the integer approach applied to a realistic scenario—a full receipt printer for the Academy shop:

In [ ]:
def pence_to_str(pennies):
    sign   = "-" if pennies < 0 else ""
    pennies = abs(pennies)
    pounds = pennies // 100
    pence  = pennies  % 100
    return f"{sign}£{pounds:,}.{pence:02d}"


# The shop's inventory — all prices in pennies
basket = [
    ("Dragon Scale Shield",   4999),   # £49.99
    ("Healing Potion x3",     1497),   # £14.97
    ("Enchanted Compass",     2350),   # £23.50
    ("Apprentice Spellbook",   850),   # £8.50
]

TAX_RATE_PCT = 20   # 20% guild tax, kept as integer percentage
DISCOUNT_PCT = 10   # 10% loyalty discount for this initiate

subtotal    = sum(price for _, price in basket)
discount    = subtotal * DISCOUNT_PCT // 100
after_disc  = subtotal - discount
tax         = after_disc * TAX_RATE_PCT // 100
total       = after_disc + tax

# --- Print the receipt ---
WIDTH = 42
print("=" * WIDTH)
print(f"{'  ACADEMY SHOP RECEIPT':^{WIDTH}}")
print("=" * WIDTH)
for name, price in basket:
    print(f"  {name:<24} {pence_to_str(price):>12}")
print("-" * WIDTH)
print(f"  {'Subtotal':<24} {pence_to_str(subtotal):>12}")
print(f"  {f'Loyalty Discount ({DISCOUNT_PCT}%)':<24} {pence_to_str(-discount):>12}")
print(f"  {'After Discount':<24} {pence_to_str(after_disc):>12}")
print(f"  {f'Guild Tax ({TAX_RATE_PCT}%)':<24} {pence_to_str(tax):>12}")
print("=" * WIDTH)
print(f"  {'TOTAL':<24} {pence_to_str(total):>12}")
print("=" * WIDTH)

---

## 🏆 The Initiate's Trial: The Currency Converter

The Academy deals with three currencies:

| Currency | Code | Pennies per unit |
| :--- | :--- | :--- |
| Gold Sovereigns | GS | 100 copper |
| Silver Marks | SM | 12 copper |
| Copper Pennies | CP | 1 copper |

Write a spell that:
1. Accepts a total in **copper pennies** as an integer.
2. Uses `//` and `%` to break it down into Gold Sovereigns, Silver Marks, and Copper Pennies (fewest coins possible, like making change).
3. Displays the result neatly.

**Example:** `1357` copper → `13 GS, 4 SM, 9 CP`  
*(13 × 100 = 1300, 4 × 12 = 48, 9 × 1 = 9 → 1300 + 48 + 9 = 1357 ✓)*

> [!TIP]
> Convert greedily: extract Gold first, then Silver from the remainder, then Copper from whatever's left.

In [ ]:
total_copper = int(input("Enter amount in copper pennies: "))

# --- YOUR CURRENCY CONVERTER HERE ---

# ------------------------------------

---

**Outstanding work, Initiate!** You now understand why money is treacherous magical territory, and you have three reliable tools to handle it safely—from the simple `round()` patch, to the elegant integer-penny method, to the industrial-grade `decimal.Decimal`. A true financial wizard always picks the right tool for the stakes involved.

[Previous Lesson: Floats](./2-3-floats.ipynb) | [Next Lesson: Decisions](./3-1-Decisions.ipynb)